# 📖 Module 02: Data Parsing & Document Loaders

## GenAI L2 Exam Preparation

**Topics Covered:**
- LangChain Document Loaders (PDF, DOCX, CSV, JSON, HTML)
- Advanced Parsers (PyMuPDF, pdfplumber, Docling, LlamaParse)
- When to use which parser
- Hands-on: Load and inspect documents

**Source Material:** Class 30 (Data Parsing for RAG), Class 32 (Data Parsing Extended)

---

## 1. Why Data Parsing Matters

Data parsing is the **first and most critical step** of a RAG pipeline. If your documents are parsed poorly, the entire pipeline suffers:

```
Bad Parsing → Bad Chunks → Bad Embeddings → Bad Retrieval → Bad Answers
```

### Challenges in Data Parsing

| Challenge | Description |
|-----------|-------------|
| **Mixed content** | Documents with text, tables, images, headers, footers |
| **Layout complexity** | Multi-column PDFs, nested structures |
| **Encoding issues** | Different character encodings across files |
| **Metadata loss** | Losing page numbers, section headings during parsing |
| **Format variety** | PDFs, DOCX, HTML, CSV, JSON — each needs different handling |

### 🎯 Exam Tip
> Data quality at ingestion is the **#1 factor** affecting RAG pipeline quality.  
> "Garbage in, garbage out" applies strongly to RAG.

## 2. LangChain Document Loaders

LangChain provides a unified `DocumentLoader` interface. Every loader returns a list of `Document` objects with:
- `page_content` — the extracted text
- `metadata` — source info (filename, page number, etc.)

### Loader Selection Guide

| File Type | Loader | Package | Best For |
|-----------|--------|---------|----------|
| **PDF** | `PyPDFLoader` | `langchain_community` | Simple text PDFs |
| **PDF** | `UnstructuredPDFLoader` | `langchain_community` | Complex layouts |
| **DOCX** | `Docx2txtLoader` | `langchain_community` | Word documents |
| **CSV** | `CSVLoader` | `langchain_community` | Tabular data |
| **JSON** | `JSONLoader` | `langchain_community` | Structured JSON |
| **HTML** | `BSHTMLLoader` | `langchain_community` | Web pages |
| **Text** | `TextLoader` | `langchain_community` | Plain text files |
| **Directory** | `DirectoryLoader` | `langchain_community` | Multiple files at once |

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()
print("✅ Environment loaded")

### 2.1 TextLoader — Plain Text Files

In [ ]:
from langchain_community.document_loaders import TextLoader

# Load a text file
loader = TextLoader("./data/sample.txt", encoding="utf-8")
documents = loader.load()

print(f"📄 Number of documents: {len(documents)}")
print(f"📝 Content preview: {documents[0].page_content[:200]}...")
print(f"📋 Metadata: {documents[0].metadata}")

### 2.2 PyPDFLoader — PDF Files

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load a PDF file (each page becomes a separate Document)
# Using class material as example
pdf_path = "../Class-29-RAG-Introduction/Class-29-handwritten-notes-04-July-2026-RAG-Intro.pdf"

try:
    loader = PyPDFLoader(pdf_path)
    pdf_docs = loader.load()
    
    print(f"📄 Total pages loaded: {len(pdf_docs)}")
    print(f"📝 Page 1 preview: {pdf_docs[0].page_content[:200]}...")
    print(f"📋 Metadata: {pdf_docs[0].metadata}")
except Exception as e:
    print(f"⚠️ Could not load PDF: {e}")
    print("This is expected if the PDF path doesn't exist")

### 2.3 CSVLoader — Tabular Data

In [ ]:
# First, create a sample CSV for demonstration
import csv
import os

csv_data = [
    ["Concept", "Description", "Category"],
    ["RAG", "Retrieval-Augmented Generation - combines retrieval with LLM generation", "Architecture"],
    ["FAISS", "Facebook AI Similarity Search - fast local vector search library", "Vector DB"],
    ["ChromaDB", "Open-source embedding database for local development", "Vector DB"],
    ["Chunking", "Splitting documents into smaller pieces for better retrieval", "Data Processing"],
    ["Embedding", "Converting text to numerical vectors that capture semantic meaning", "Representation"],
]

csv_path = "./data/concepts.csv"
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(csv_data)

print(f"✅ Created sample CSV at {csv_path}")

In [ ]:
from langchain_community.document_loaders import CSVLoader

# Each row becomes a separate Document
loader = CSVLoader("./data/concepts.csv")
csv_docs = loader.load()

print(f"📄 Number of documents (rows): {len(csv_docs)}")
print(f"\n📝 First document:")
print(csv_docs[0].page_content)
print(f"\n📋 Metadata: {csv_docs[0].metadata}")

### 2.4 JSONLoader — Structured JSON

In [ ]:
# Create sample JSON
import json

json_data = {
    "topics": [
        {
            "name": "RAG Pipeline",
            "description": "End-to-end pipeline for retrieval-augmented generation",
            "components": ["Loader", "Splitter", "Embeddings", "Vector Store", "Retriever", "LLM"]
        },
        {
            "name": "Vector Database",
            "description": "Database optimized for storing and searching vector embeddings",
            "components": ["FAISS", "ChromaDB", "Pinecone", "Qdrant"]
        }
    ]
}

json_path = "./data/topics.json"
with open(json_path, 'w') as f:
    json.dump(json_data, f, indent=2)

print(f"✅ Created sample JSON at {json_path}")

In [ ]:
from langchain_community.document_loaders import JSONLoader

# jq_schema specifies which field to extract
loader = JSONLoader(
    file_path="./data/topics.json",
    jq_schema=".topics[].description",  # Extract descriptions
    text_content=False
)
json_docs = loader.load()

print(f"📄 Number of documents: {len(json_docs)}")
for doc in json_docs:
    print(f"📝 {doc.page_content}")

### 2.5 DirectoryLoader — Load Multiple Files

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

# Load all text files from a directory
loader = DirectoryLoader(
    "./data/",
    glob="*.txt",  # File pattern to match
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
dir_docs = loader.load()

print(f"📄 Total documents loaded from directory: {len(dir_docs)}")
for doc in dir_docs:
    print(f"  📁 Source: {doc.metadata.get('source', 'unknown')}")

## 3. Advanced Parsers

When basic LangChain loaders aren't enough, use advanced parsers:

| Parser | Best For | Advantage | Disadvantage |
|--------|----------|-----------|-------------|
| **PyMuPDF** | PDF text extraction | Fast, preserves layout | Struggles with complex tables |
| **pdfplumber** | PDF tables | Excellent table extraction | Slower than PyMuPDF |
| **Docling** | Multi-format | Handles PDF, DOCX, HTML, images | Requires `docling` package |
| **LlamaParse** | Complex PDFs | Best for complex layouts, tables, images | Cloud API (requires API key) |
| **Unstructured** | Any format | Wide format support | Can be slow, heavy dependency |

In [ ]:
# PyMuPDF Example (pip install pymupdf)
# This is a popular alternative to PyPDFLoader

try:
    import fitz  # PyMuPDF
    
    pdf_path = "../Class-29-RAG-Introduction/Class-29-handwritten-notes-04-July-2026-RAG-Intro.pdf"
    doc = fitz.open(pdf_path)
    
    print(f"📄 PDF has {len(doc)} pages")
    
    # Extract text from first page
    page = doc[0]
    text = page.get_text()
    print(f"📝 Page 1 text (first 300 chars):\n{text[:300]}")
    
    doc.close()
except ImportError:
    print("⚠️ PyMuPDF not installed. Run: pip install pymupdf")
except Exception as e:
    print(f"⚠️ Could not load PDF: {e}")

## 4. The Document Object

Every loader returns `Document` objects. Understanding their structure is essential:

```python
from langchain_core.documents import Document

doc = Document(
    page_content="This is the actual text content...",
    metadata={
        "source": "path/to/file.pdf",
        "page": 0,
        "author": "John Doe",
        "custom_field": "any value you want"
    }
)
```

### Metadata Best Practices
- Always include **source** (for citation/attribution)
- Include **page number** for PDFs (helps with debugging)
- Add **custom metadata** for filtering (e.g., department, date, category)

### 🎯 Exam Tip
> Metadata is crucial for:
> 1. **Source attribution** — telling users where the answer came from
> 2. **Metadata filtering** — narrowing retrieval to specific documents/sections
> 3. **Debugging** — tracing back to the original source when retrieval is poor

## 5. Loader Selection Decision Tree

```
What type of file do you have?
│
├─ PDF
│  ├─ Simple text PDF → PyPDFLoader
│  ├─ PDF with tables → pdfplumber
│  ├─ Complex layout (multi-column, images) → LlamaParse or Docling
│  └─ Need speed → PyMuPDF (fitz)
│
├─ Word Document (.docx)
│  └─ Docx2txtLoader (simple) or UnstructuredWordDocumentLoader (complex)
│
├─ CSV / Excel
│  ├─ CSV → CSVLoader
│  └─ Excel → UnstructuredExcelLoader
│
├─ JSON
│  └─ JSONLoader (with jq_schema for field selection)
│
├─ HTML / Web Pages
│  └─ BSHTMLLoader (BeautifulSoup-based)
│
├─ Plain Text
│  └─ TextLoader
│
└─ Multiple files in a directory
   └─ DirectoryLoader (with appropriate loader_cls)
```

### 🎯 Exam Tip
> The exam may give you a scenario and ask which loader to use.  
> Key pattern: **Complex PDFs with tables** → pdfplumber or LlamaParse  
> Key pattern: **Simple text extraction** → PyPDFLoader or TextLoader

## 🧠 Self-Assessment Quiz

---

**Q1.** What two attributes does every LangChain `Document` object have?

<details>
<summary>Click for Answer</summary>

1. `page_content` — the text content  
2. `metadata` — dictionary of metadata (source, page number, etc.)
</details>

---

**Q2.** You need to extract tables from a PDF with complex formatting. Which parser would you choose?

<details>
<summary>Click for Answer</summary>

**pdfplumber** — it's specifically designed for table extraction from PDFs. For very complex layouts with images, **LlamaParse** or **Docling** would be even better.
</details>

---

**Q3.** What is the `glob` parameter in `DirectoryLoader` used for?

<details>
<summary>Click for Answer</summary>

The `glob` parameter specifies a **file pattern** to match files in the directory. For example, `"*.pdf"` loads only PDF files, `"**/*.txt"` loads all text files recursively.
</details>

---

**Q4.** Why is metadata important in a RAG pipeline?

<details>
<summary>Click for Answer</summary>

Metadata is important for:  
1. **Source attribution** — citing where the answer came from  
2. **Metadata filtering** — narrowing search to specific documents/categories  
3. **Debugging** — tracing retrieval issues back to source documents
</details>

---

**Q5.** What is the `jq_schema` parameter in `JSONLoader`?

<details>
<summary>Click for Answer</summary>

The `jq_schema` parameter specifies a **jq filter expression** to select which fields from the JSON to extract as document content. For example, `.data[].text` extracts the `text` field from each item in the `data` array.
</details>

---

## ✅ Module 2 Complete!

**Key Takeaways:**
1. Data parsing quality directly impacts entire RAG pipeline performance
2. LangChain provides loaders for all common file types
3. Every loader returns `Document(page_content, metadata)` objects
4. Choose parsers based on document complexity (simple → PyPDF, complex → LlamaParse)
5. Metadata is essential for source attribution and filtering

**Next:** [Module 03 — Chunking Strategies](./03_Chunking_Strategies.ipynb)